# small128 iter-2 — scale the gate-validated loop (value-head crisis corpus)

**Base: `small128_vh1`** (= gate3/epoch_1; 5k bar: **mean 13,080 / P50 9,323 / P5 1,222 / <1000 3.5%**).
**Corpus `iter2_mix.pt` = 6,355,150 states**: 2,508,531 NEW signal (7,708 head-guided crisis replays
of vh1's own deaths: recovery 15-back@1200 40% escape / prevention 30-back@800 84% escape, q=2.0,
value_head_small128_vh1, all C++-mined) + 3,846,619 rehearsal (full distill corpus). 39% signal
(gate-3's proven mix was 25% — the signal corpus outgrew the rehearsal pool; ep1 gate protects).

**Recipe = the gate-3 winner (HISTORY 177), scaled:** warm-start, blend 0.5 hard-CE, T=1.0, no dw,
lr 1e-4, bs 4096, seeded. **Pick by ep1-first floor gate** (every winning run here improved by ep1).

**Upload to `MyDrive/alphatrain/`:**
1. `colorlines_pillar3d_v3.tar.gz` (521,888 B — REBUILT, has `--seed`)
2. `iter2_mix.pt.gz` (340,318,726 B)
3. `small128_vh1.pt` (36,156,933 B)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time
DRIVE='/content/drive/MyDrive/alphatrain'
!cp {DRIVE}/colorlines_pillar3d_v3.tar.gz /content/
!cd /content && tar xzf colorlines_pillar3d_v3.tar.gz
os.makedirs('/content/alphatrain/data', exist_ok=True)
t0=time.time()
!cp {DRIVE}/iter2_mix.pt.gz /content/iter2_mix.pt.gz
gz=os.path.getsize('/content/iter2_mix.pt.gz'); print(f'.gz: {gz:,} bytes')
assert gz == 340_318_726, f'.gz truncated! got {gz}; re-upload iter2_mix.pt.gz'
!gunzip -t /content/iter2_mix.pt.gz && echo '.gz integrity OK'
!gzip -dc /content/iter2_mix.pt.gz > /content/alphatrain/data/iter2_mix.pt
pt=os.path.getsize('/content/alphatrain/data/iter2_mix.pt')
assert pt == 1_010_472_171, f'.pt size wrong! got {pt}'
print(f'corpus: {pt/1e9:.2f} GB, 6,355,150 states ({time.time()-t0:.0f}s)')
!rm /content/iter2_mix.pt.gz
!cp {DRIVE}/small128_vh1.pt /content/alphatrain/data/
assert os.path.getsize('/content/alphatrain/data/small128_vh1.pt') == 36_156_933
!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g=torch.cuda.get_device_properties(0); print(f'GPU {torch.cuda.get_device_name(0)} | {g.total_memory/1e9:.0f} GB')

In [ ]:
# ===== CONFIG (gate-3 winning recipe, scaled corpus) =====
CHANNELS = 128
EPOCHS   = 4         # gate at ep1; per-epoch saves. Winning epochs are EARLY here.
BATCH    = 4096
LR       = 1e-4
T        = 1.0
DW       = 0
BLEND    = 0.5
SEED     = 42
RUN      = "small128_iter2"
print(f'RUN={RUN} epochs={EPOCHS} batch={BATCH} lr={LR} T={T} dw={DW} blend={BLEND} seed={SEED}')

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/iter2_mix.pt \
    --resume alphatrain/data/small128_vh1.pt --warm-start \
    --channels {CHANNELS} --seed {SEED} --amp --compile \
    --epochs {EPOCHS} --batch-size {BATCH} --lr {LR} --warmup-epochs 1 \
    --target-temperature {T} --decisiveness-power {DW} --blend-alpha {BLEND} \
    --copy-to /content/drive/MyDrive/alphatrain/{RUN}_best.pt \
    --save-dir /content/checkpoints/{RUN} 2>&1 | tee /content/{RUN}_train.log
# val is informational only — the gate is GAMEPLAY (ep1-first, floor-first).

In [ ]:
import shutil, os, glob
DRIVE='/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob(f'/content/checkpoints/{RUN}/epoch_*.pt')):
    dst=f'{DRIVE}/{RUN}_{os.path.basename(f)}'; shutil.copy(f,dst); print('Saved', dst)
for f in ['best.pt','latest.pt']:
    s=f'/content/checkpoints/{RUN}/{f}'
    if os.path.exists(s): shutil.copy(s,f'{DRIVE}/{RUN}_{f}'); print('Saved', f'{DRIVE}/{RUN}_{f}')

## Gate protocol (M5, C++ eval — download epochs as they save)

```bash
python -m alphatrain.inference_cpp.export_ts --model alphatrain/data/small128_iter2_epoch_1.pt
cd alphatrain/inference_cpp
./build/eval --model data/policy_ts.pt --device mps --seed-start 775000 --seed-end 775500 --batch 500
```
1. **ep1 first** (rejection gate, floor-first) vs the vh1 bar: 5k mean 13,080 / P50 9,323 /
   P5 1,222 / P10 1,889 / <1000 3.5%.
2. ep1 ≥ bar → eval ep2/3/4, pick by floor, then **5k confirm** (`--seed-end 780000`) of the pick
   AGAINST vh1's 5k numbers (500 seeds mislead close calls — gate-3's median read −6% at 500
   and +4.6% at 5k).
3. If it clears: name it `small128_vh2`, retrain the value head on ITS backbone (8 min, HISTORY
   158), re-export fused module, and iteration 3 is a single mining command.
4. ep1 regression → fallback: rebuild mix at 25% signal (subsample crisis to 1.28M) and rerun.